In [3]:
# === Re-test held-out set (th=0.05, no morph, no size filter) ===
from pathlib import Path
import os, time, json, csv, gc
import numpy as np
import tensorflow as tf
import nibabel as nib

# ---------------------- USER PATHS ----------------------
RUN_DIR = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011")  # training run folder
TEST_DIR = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_lores")
MODULE_PATH = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
# --------------------------------------------------------

# ---- Inference recipe (picked from your sweep: best macro-hard) ----
THRESH   = 0.05
MIN_SIZE = 0
DO_CLOSE = False

RUN_TAG   = f"{TEST_DIR.name}_t{THRESH:.2f}_min{MIN_SIZE}_close{int(DO_CLOSE)}"
OUT_DIR   = RUN_DIR / f"test_preds_{RUN_TAG}"
ART_DIR   = RUN_DIR / "test_artifacts"
CSV_PATH  = ART_DIR / f"test_cases_{RUN_TAG}.csv"
JSON_PATH = ART_DIR / f"test_summary_{RUN_TAG}.json"
OUT_DIR.mkdir(parents=True, exist_ok=True); ART_DIR.mkdir(parents=True, exist_ok=True)

# ---- helpers ----
def clean_base(p: Path) -> str:
    name = p.name
    if name.endswith(".nii.gz"):
        return name[:-7]
    return p.stem

def save_like(ref_path: Path, array: np.ndarray, out_path: Path, dtype=None):
    """Save `array` as NIfTI using `ref_path`'s affine/header."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    ref = nib.load(str(ref_path))
    data = np.asarray(array, dtype=(dtype if dtype is not None else np.float32))
    nii = nib.Nifti1Image(np.ascontiguousarray(data), ref.affine, ref.header.copy())
    nib.save(nii, str(out_path))

# --- import your training module & custom layers ---
import importlib.util
spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

# Single-device default strategy for inference
seg.strategy = tf.distribute.get_strategy()

# --- load the full Keras model from this run ---
keras_models = sorted((RUN_DIR / "models").glob("smart_sota_dynamic_*.keras"))
if not keras_models:
    raise FileNotFoundError(f"No .keras model found in {RUN_DIR/'models'}")
MODEL_PATH = keras_models[0]
print("Loading model:", MODEL_PATH.name)

from keras.saving import load_model
m = load_model(
    MODEL_PATH,
    compile=False,
    custom_objects={
        "ResidualConvBlock": seg.ResidualConvBlock,
        "VisionMambaBlock": seg.VisionMambaBlock,
        "SAM2Attention": seg.SAM2Attention,
        "CombinedLoss": seg.CombinedLoss,
        "dice_coefficient": seg.dice_coefficient,
        "dice_loss": seg.dice_loss,
        "boundary_loss": seg.boundary_loss,
    },
)
INPUT_SHAPE = tuple(m.input_shape[1:])
print("Model INPUT_SHAPE:", INPUT_SHAPE)

# --- simple post-processing (threshold + optional morph + min-size) ---
from scipy.ndimage import label, generate_binary_structure, binary_closing
def postproc(prob, th=THRESH, min_size=MIN_SIZE, do_close=DO_CLOSE):
    b = (prob >= th)
    if do_close:
        b = binary_closing(b, structure=generate_binary_structure(3,1))
    if min_size > 0:
        s = generate_binary_structure(3,1)
        lab, n = label(b, structure=s)
        if n > 0:
            sizes = np.bincount(lab.ravel())
            keep = np.zeros_like(b, bool)
            for cid, sz in enumerate(sizes):
                if cid and sz >= min_size:
                    keep |= (lab == cid)
            b = keep
    return b.astype(np.uint8)

# --- dataset from TEST_DIR (single-folder) ---
cfg = seg.DynamicTrainingConfig(
    DATA_DIR=TEST_DIR,
    IMAGES_DIR=None, MASKS_DIR=None,
    INPUT_SHAPE=INPUT_SHAPE,
    VALIDATION_SPLIT=0.10,       # unused for testing; required by dataclass
    BATCH_SIZE=1,
    MODEL_DIR=RUN_DIR / "models",
    CALLBACKS_DIR=RUN_DIR / "callbacks",
)
pairs, lesion_presence = seg.load_generic_dataset(cfg)
print(f"Pairs: {len(pairs)} | % non-empty masks: {100*np.mean(lesion_presence):.1f}%")

# --- dice helpers ---
def dice_soft(y, p, eps=1e-8):
    inter = float(np.sum(y * p, dtype=np.float64))
    denom = float(np.sum(y, dtype=np.float64) + np.sum(p, dtype=np.float64) + eps)
    return 2.0 * inter / denom, inter

def dice_hard(y, b, eps=1e-8):
    inter = float(np.sum((y > 0) & (b > 0), dtype=np.float64))
    denom = float(np.sum(y > 0, dtype=np.float64) + np.sum(b > 0, dtype=np.float64) + eps)
    return 2.0 * inter / denom, inter

# --- evaluation loop ---
rows = []
sum_inter_soft = 0.0
sum_y = 0.0
sum_p = 0.0
sum_inter_hard = 0.0
sum_b = 0.0

for i, (img_p, msk_p) in enumerate(pairs, 1):
    # load & shape
    img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    y   = seg._load_and_preprocess_mask(str(msk_p),  INPUT_SHAPE[:-1]).astype(np.float32)
    x   = np.zeros((1,*INPUT_SHAPE), np.float32); x[0,...,0] = img

    # predict
    p = m.predict(x, verbose=0)[0,...,0].astype(np.float32)
    b = postproc(p, THRESH, MIN_SIZE, DO_CLOSE)

    # metrics
    ds, inter_s = dice_soft(y, p)
    dh, inter_h = dice_hard(y, b)
    vy = float(np.sum(y > 0, dtype=np.float64))
    vp = float(np.sum(p >= THRESH, dtype=np.float64))

    # micro accumulators
    sum_inter_soft += inter_s
    sum_y          += vy
    sum_p          += float(np.sum(p, dtype=np.float64))
    sum_inter_hard += inter_h
    sum_b          += float(np.sum(b > 0, dtype=np.float64))

    # save predictions
    base = clean_base(img_p).replace("_T1w", "")
    save_like(msk_p, p, OUT_DIR / f"{base}_soft.nii.gz", dtype=np.float32)
    save_like(msk_p, b, OUT_DIR / f"{base}_hard.nii.gz", dtype=np.uint8)

    rows.append({
        "case": base,
        "soft_dice": ds,
        "hard_dice": dh,
        "lesion_voxels": int(vy),
        "pred_voxels@th": int(vp),
        "inter_soft_vox": int(inter_s),
        "inter_hard_vox": int(inter_h),
    })
    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] last case soft={ds:.4f} hard@{THRESH:.2f}={dh:.4f}", flush=True)

# --- aggregate & report ---
macro_soft = float(np.mean([r["soft_dice"] for r in rows])) if rows else 0.0
macro_hard = float(np.mean([r["hard_dice"] for r in rows])) if rows else 0.0
micro_soft = float(2.0 * sum_inter_soft / (sum_y + sum_p + 1e-8)) if (sum_y + sum_p) > 0 else 0.0
micro_hard = float(2.0 * sum_inter_hard / (sum_y + sum_b + 1e-8)) if (sum_y + sum_b) > 0 else 0.0

print("\n=== HELD-OUT RESULTS (chosen recipe) ===")
print(f"Per-case (macro) soft Dice      : {macro_soft:.4f}")
print(f"Per-case (macro) hard Dice @{THRESH:.2f}: {macro_hard:.4f}")
print(f"Global (micro) soft Dice        : {micro_soft:.4f}")
print(f"Global (micro) hard Dice @{THRESH:.2f}: {micro_hard:.4f}")
print(f"Saved predictions -> {OUT_DIR}")

# --- artifacts ---
with open(CSV_PATH, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)

summary = {
    "threshold": THRESH,
    "min_size": MIN_SIZE,
    "closing": DO_CLOSE,
    "n_cases": len(rows),
    "macro_soft": macro_soft,
    "macro_hard": macro_hard,
    "micro_soft": micro_soft,
    "micro_hard": micro_hard,
    "run_dir": str(RUN_DIR),
    "test_dir": str(TEST_DIR),
    "out_dir": str(OUT_DIR),
}
with open(JSON_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("\nWrote per-case CSV  ->", CSV_PATH)
print("Wrote summary JSON  ->", JSON_PATH)

# hygiene
tf.keras.backend.clear_session(); gc.collect();


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


2025-11-06 14:14:16,446 - tensorflow - INFO - Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
2025-11-06 14:14:16,450 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-06 14:14:16,450 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-06 14:14:16,451 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 2


Strategy: MirroredStrategy
Loading model: smart_sota_dynamic_20251105_165011.keras


2025-11-06 14:14:17,458 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2025-11-06 14:14:17,459 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=1.82GB | GPU mem tracking failed | Disk: 1244.4GB free
2025-11-06 14:14:17,474 - SmartSOTA_Dynamic - INFO - 📁 Single-folder mode: 396 images, 396 masks in /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_lores
2025-11-06 14:14:17,474 - SmartSOTA_Dynamic - INFO - Found 396 image files and 396 mask files


Model INPUT_SHAPE: (192, 224, 192, 1)


2025-11-06 14:14:44,668 - SmartSOTA_Dynamic - INFO - 📊 Created 198 image–mask pairs
2025-11-06 14:14:44,669 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-11-06 14:14:44,670 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=1.84GB | GPU mem tracking failed | Disk: 1244.4GB free


Pairs: 198 | % non-empty masks: 100.0%
[10/198] last case soft=0.3507 hard@0.05=0.3650
[20/198] last case soft=0.6024 hard@0.05=0.6154
[30/198] last case soft=0.2501 hard@0.05=0.2531
[40/198] last case soft=0.2784 hard@0.05=0.3124
[50/198] last case soft=0.7920 hard@0.05=0.7912
[60/198] last case soft=0.0708 hard@0.05=0.0708
[70/198] last case soft=0.0206 hard@0.05=0.0200
[80/198] last case soft=0.0010 hard@0.05=0.0015
[90/198] last case soft=0.0641 hard@0.05=0.0625
[100/198] last case soft=0.3911 hard@0.05=0.3909
[110/198] last case soft=0.3634 hard@0.05=0.3610
[120/198] last case soft=0.7183 hard@0.05=0.7180
[130/198] last case soft=0.0019 hard@0.05=0.0020
[140/198] last case soft=0.1152 hard@0.05=0.1168
[150/198] last case soft=0.0131 hard@0.05=0.0139
[160/198] last case soft=0.0001 hard@0.05=0.0000
[170/198] last case soft=0.0000 hard@0.05=0.0000
[180/198] last case soft=0.0001 hard@0.05=0.0000
[190/198] last case soft=0.0037 hard@0.05=0.0044
[198/198] last case soft=0.3352 hard@0.